# Flu Shot Learning: LightGBM with Multilabel Stratified K-Fold

This notebook trains one LightGBM binary classifier per target (Binary Relevance) for the DrivenData Flu Shot Learning competition. It reuses the engineered survey features from the ensemble experiment, leaves numeric missing values for LightGBM to handle natively, and passes pandas categorical columns through LightGBM's native categorical support.

**Validation:** five-fold `MultilabelStratifiedKFold`, stratifying the H1N1 and seasonal vaccine labels jointly.

**Metric:** mean ROC AUC across `h1n1_vaccine` and `seasonal_vaccine`.

In [1]:
from pathlib import Path
import importlib.util
import sys

packages_to_install = {
    'lightgbm': 'lightgbm',
    'iterstrat': 'iterative-stratification',
}
missing_packages = [
    package_name
    for module_name, package_name in packages_to_install.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    !{sys.executable} -m pip install -q {' '.join(missing_packages)}

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
pd.set_option('display.max_columns', 100)


## 1. Load competition data

In [2]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),

    ]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find training_set_features.csv. Set DATA_DIR to the competition data folder.'
    )

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')

assert train_features['respondent_id'].is_unique
assert test_features['respondent_id'].is_unique
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert set(train_features['respondent_id']).isdisjoint(test_features['respondent_id'])

y = train_labels[TARGETS].astype('int8').copy()
print(f'Using: {DATA_DIR.resolve()}')
print(f'Train: {train_features.shape}; test: {test_features.shape}')
display(y.mean().rename('positive_rate').to_frame())

Using: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset
Train: (26707, 36); test: (26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


## 2. Reuse leakage-free engineered features

These summaries use only respondent features. Numeric NaNs are deliberately not filled: LightGBM learns a default direction for missing numeric values.

In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)

    behavior_columns = [c for c in result.columns if c.startswith('behavioral_')]
    result['behavior_score'] = result[behavior_columns].sum(axis=1, min_count=1)
    result['behavior_count_available'] = result[behavior_columns].notna().sum(axis=1)

    opinion_columns = [c for c in result.columns if c.startswith('opinion_')]
    result['opinion_score'] = result[opinion_columns].mean(axis=1)
    risk_columns = [c for c in opinion_columns if '_risk' in c]
    result['opinion_risk_score'] = result[risk_columns].mean(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)

    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        if column in result:
            result[column + '_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Use shared category levels so train and test have identical categorical metadata.
for column in categorical_columns:
    levels = pd.concat([X[column], X_test[column]], axis=0).dropna().unique()
    category_dtype = pd.api.types.CategoricalDtype(categories=levels)
    X[column] = X[column].astype(category_dtype)
    X_test[column] = X_test[column].astype(category_dtype)

print(f'Features: {X.shape[1]}')
print(f'Categorical features: {len(categorical_columns)}')
print(f'Numeric features: {X.shape[1] - len(categorical_columns)}')

Features: 46
Categorical features: 12
Numeric features: 34


## 3. Five-fold multilabel-stratified binary relevance models

`MultilabelStratifiedKFold` preserves the joint distribution of both target columns without concatenating labels into a string. Each target receives its own LightGBM model in every fold.

In [4]:
def make_model(target, seed):
    return LGBMClassifier(
        objective='binary',
        n_estimators=2500,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

splitter = MultilabelStratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
folds = list(splitter.split(X, y))
oof = pd.DataFrame(index=X.index, columns=TARGETS, dtype=float)
test_predictions = pd.DataFrame(0.0, index=X_test.index, columns=TARGETS)
best_iterations = {target: [] for target in TARGETS}

for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
    print(f'Fold {fold_number}/{N_SPLITS}')
    for target in TARGETS:
        model = make_model(target, RANDOM_STATE + fold_number)
        model.fit(
            X.iloc[train_index], y.iloc[train_index][target],
            eval_set=[(X.iloc[valid_index], y.iloc[valid_index][target])],
            eval_metric='auc',
            categorical_feature=categorical_columns,
            callbacks=[early_stopping(150, verbose=False), log_evaluation(0)],
        )
        oof.loc[valid_index, target] = model.predict_proba(X.iloc[valid_index])[:, 1]
        test_predictions[target] += model.predict_proba(X_test)[:, 1] / N_SPLITS
        best_iterations[target].append(model.best_iteration_)

auc_by_target = {
    target: roc_auc_score(y[target], oof[target])
    for target in TARGETS
}
print('\nOOF ROC AUC by target:')
display(pd.Series(auc_by_target).rename('roc_auc').to_frame())
print(f"Mean OOF ROC AUC: {np.mean(list(auc_by_target.values())):.5f}")
print('Best iterations:', {target: int(np.median(values)) for target, values in best_iterations.items()})

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

OOF ROC AUC by target:


,roc_auc
h1n1_vaccine,0.870347
seasonal_vaccine,0.863831


Mean OOF ROC AUC: 0.86709
Best iterations: {'h1n1_vaccine': 208, 'seasonal_vaccine': 311}


## 4. Save OOF predictions and competition submission

The submission contains probabilities in the original test respondent order.

In [5]:
output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'respondent_id': test_features['respondent_id']})
submission[TARGETS] = test_predictions[TARGETS].to_numpy()
submission_path = output_dir / 'submission_lightgbm_multilabel.csv'
oof_path = output_dir / 'oof_lightgbm_multilabel.csv'
submission.to_csv(submission_path, index=False)
oof.assign(respondent_id=train_features['respondent_id']).to_csv(oof_path, index=False)

print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())

Saved submission: /kaggle/working/outputs/submission_lightgbm_multilabel.csv
Saved OOF predictions: /kaggle/working/outputs/oof_lightgbm_multilabel.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.107259,0.287066
1,26708,0.043536,0.051078
2,26709,0.153588,0.610666
3,26710,0.665341,0.886461
4,26711,0.229763,0.456641
